In [1]:
# =========================================================
# CELL 1: CÀI ĐẶT & CẤU HÌNH ĐƯỜNG DẪN
# =========================================================
!pip install -q "huggingface_hub>=0.26.2" pyyaml

In [2]:
import os
import json
import random
import shutil
import re
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
import yaml
from huggingface_hub import hf_hub_download

# --- INPUT DATA ---
GOLD_DIR = Path("/kaggle/input/datasets/notpitomon/htd-better-gold-dataset/Merged_Rukopys_V2/train")
SILVER_DIR = Path("None")

# --- K-FOLD CONFIG ---
VAL_FOLD = 5 # Co the chinh thanh 1, 2, 3, 4, hoac 5. Fold nay se dung lam tap VAL, 4 fold kia lam TRAIN.
# Duong dan thu muc chua 5 file JSONL da cat (metadata_part1.jsonl, part2, part3, part4, part5).
# Buoc nay ban se tu Upload dataset Kaggle chua 5 file nay va sua duong dan vao day:
META_PARTS_DIR = Path("/kaggle/input/datasets/notpitomon/htd-better-gold-dataset/MRV2.1_Metadata/MRV2.1_Metadata")

# --- MODEL WEIGHTS ---
# Set this to a valid .pt file in /kaggle/input when available.
BASE_MODEL_PATH = ""

# --- OUTPUT YOLO DATASET ---
OUT_ROOT = Path("/kaggle/working/layout_data/rukopys_v3")
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

BBOX_FORMAT = "xyxy"

# --- CLASS MAP ---
TYPE2ID = {
    "handwritten": 0, "printed": 1, "formula": 2, "table": 3,
    "annotation": 4, "image": 5, "graph": 6,
}
ID2TYPE = {v: k for k, v in TYPE2ID.items()}

# Silver supplement settings (Neu thay can thiet co the sua USE_SILVER = True)
USE_SILVER = False
MAX_SILVER_IMAGES = 1200
SEED = 42

random.seed(SEED)

print("✅ Config xong.")
print(f"VAL_FOLD   = {VAL_FOLD}")
print(f"META_PARTS = {META_PARTS_DIR}")

✅ Config xong.
VAL_FOLD   = 5
META_PARTS = /kaggle/input/datasets/notpitomon/htd-better-gold-dataset/MRV2.1_Metadata/MRV2.1_Metadata


In [3]:
# =========================================================
# CELL 2: HELPER FUNCTIONS
# =========================================================

def load_metadata(jsonl_path):
    data = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def clip(val, min_val, max_val):
    return max(min_val, min(val, max_val))

def region_to_yolo_line(region, img_w, img_h):
    cls_name = region.get("type", "")
    if cls_name not in TYPE2ID:
        return None
    cls_id = TYPE2ID[cls_name]
    
    bbox = region.get("bbox")
    if not bbox or len(bbox) != 4:
        return None
        
    # bbox format từ metadata là: [x_min, y_min, x_max, y_max] (xyxy)
    x1, y1, x2, y2 = bbox
    
    # Tính width và height
    w = x2 - x1
    h = y2 - y1
    
    # Tính tâm bbox (center x, center y)
    cx = x1 + w / 2
    cy = y1 + h / 2
    
    # Chuẩn hoá (Normalize to 0-1)
    cx_norm = clip(cx, 0, img_w) / img_w
    cy_norm = clip(cy, 0, img_h) / img_h
    w_norm  = clip(w, 0, img_w) / img_w
    h_norm  = clip(h, 0, img_h) / img_h
    
    if w_norm <= 0 or h_norm <= 0:
        return None
        
    return f"{cls_id} {cx_norm:.6f} {cy_norm:.6f} {w_norm:.6f} {h_norm:.6f}"

def safe_link(src: Path, dst: Path):
    if not dst.exists():
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copy2(src, dst)

In [4]:
# =========================================================
# CELL 3: LOAD K-FOLD METADATA
# =========================================================
print(f"🔄 ĐANG CHẠY K-FOLD CROSS VALIDATION - FOLD LÀM VAL: {VAL_FOLD}")

train_metadata = []
val_metadata = []

# Đọc 5 file JSONL đã chia
for i in range(1, 6):
    candidates = [
        META_PARTS_DIR / f"MRV2.1_part{i}.jsonl",
        META_PARTS_DIR / f"metadata_part{i}.jsonl",
        Path(f"Dataset/MRV2.1_part{i}.jsonl"),
        Path(f"Dataset/metadata_part{i}.jsonl"),
    ]

    meta_path = next((p for p in candidates if p.exists()), None)
    if meta_path is None:
        print(f"❌ LỖI: Không tìm thấy file part {i} trong {META_PARTS_DIR} hoặc Dataset/")
        continue

    part_data = load_metadata(meta_path)
    if i == VAL_FOLD:
        val_metadata.extend(part_data)
        print(f"   => Đã load Fold {i} làm tập VAL ({len(part_data)} ảnh)")
    else:
        train_metadata.extend(part_data)
        print(f"   => Đã load Fold {i} làm tập TRAIN ({len(part_data)} ảnh)")

print(f"\nGold records (TRAIN): {len(train_metadata)}")
print(f"Gold records (VAL): {len(val_metadata)}")

if USE_SILVER and (SILVER_DIR / "metadata.jsonl").exists():
    silver_metadata = load_metadata(SILVER_DIR / "metadata.jsonl")
    print(f"Silver records: {len(silver_metadata)}")
else:
    silver_metadata = []
    print("Silver records: 0")

🔄 ĐANG CHẠY K-FOLD CROSS VALIDATION - FOLD LÀM VAL: 5
   => Đã load Fold 1 làm tập TRAIN (553 ảnh)
   => Đã load Fold 2 làm tập TRAIN (552 ảnh)
   => Đã load Fold 3 làm tập TRAIN (553 ảnh)
   => Đã load Fold 4 làm tập TRAIN (552 ảnh)
   => Đã load Fold 5 làm tập VAL (552 ảnh)

Gold records (TRAIN): 2210
Gold records (VAL): 552
Silver records: 0


In [5]:
# =========================================================
# CELL 4: CHỌN SILVER BỔ TRỢ (CHỈ DÀNH CHO TRAIN NẾU ĐƯỢC BẬT)
# =========================================================
def count_classes_in_record(d):
    c = Counter()
    for r in d.get("regions", []):
        t = r.get("type")
        if t in TYPE2ID:
            c[t] += 1
    return c

selected_silver = []

if silver_metadata and USE_SILVER:
    scored = []
    for d in silver_metadata:
        regions = d.get("regions", [])
        if not regions:
            continue

        c = count_classes_in_record(d)
        total = sum(c.values())
        if total == 0:
            continue

        head = c["handwritten"] + c["formula"]
        tail = c["image"] + c["graph"]

        rare_ratio = tail / total
        dominant_ratio = head / total

        score = (
            c["graph"] * 5.0 +
            c["image"] * 4.0 +
            c["table"] * 1.5 +
            c["annotation"] * 1.0 +
            rare_ratio * 10.0 -
            dominant_ratio * 6.0 -
            max(0, c["handwritten"] - 20) * 0.15 -
            max(0, c["formula"] - 10) * 0.20
        )

        if dominant_ratio >= 0.8 and tail <= 1:
            continue

        scored.append((score, d, c))

    scored.sort(key=lambda x: x[0], reverse=True)
    selected_silver = [x[1] for x in scored[:MAX_SILVER_IMAGES]]

    aggs = Counter()
    for _, _, c in scored[:MAX_SILVER_IMAGES]:
        aggs.update(c)

    print(f"Selected silver images: {len(selected_silver)}")
else:
    print("Không dùng silver bổ trợ.")


Không dùng silver bổ trợ.


In [6]:
# =========================================================
# CELL 5: BUILD YOLO DATASET (TRAIN / VAL)
# =========================================================
train_files = []
val_files = []

gold_train_count = 0
gold_val_count = 0
silver_train_count = 0

bad_records = 0
class_counter_train = Counter()
class_counter_val = Counter()

def process_record_to_yolo(d, image_root: Path, split: str):
    global bad_records

    fname = Path(d["file_name"]).name 
    img_w, img_h = d["image_width"], d["image_height"]
    regions = d.get("regions", [])

    if not regions:
        bad_records += 1
        return False

    in_path = image_root / fname
    if not in_path.exists():
        bad_records += 1
        return False

    out_img = OUT_ROOT / "images" / fname
    safe_link(in_path, out_img)

    label_lines = []
    local_counter = Counter()

    for r in regions:
        line = region_to_yolo_line(r, img_w, img_h)
        if line is not None:
            label_lines.append(line)
            local_counter[r["type"]] += 1

    if not label_lines:
        bad_records += 1
        return False

    with open(OUT_ROOT / "labels" / f"{Path(fname).stem}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(label_lines))

    if split == "train":
        train_files.append(fname)
        class_counter_train.update(local_counter)
    else:
        val_files.append(fname)
        class_counter_val.update(local_counter)

    return True

# -----------------------
# XỬ LÝ TRAIN
# -----------------------
for d in tqdm(train_metadata, desc="Processing GOLD TRAIN"):
    # Ảnh _aug vẫn được vào Train thoải mái
    ok = process_record_to_yolo(d, GOLD_DIR / "images", "train")
    if ok:
        gold_train_count += 1

# -----------------------
# XỬ LÝ VAL
# -----------------------
for d in tqdm(val_metadata, desc="Processing GOLD VAL"):
    fname = Path(d["file_name"]).name
    # KHI VALIDATION THÌ TUYỆT ĐỐI NGĂN CHẶN CÁC FILE GENERATED AUGMENTATION lọt vào để tránh ảo tưởng
    if "_aug" in fname:
        continue
    ok = process_record_to_yolo(d, GOLD_DIR / "images", "val")
    if ok:
        gold_val_count += 1

# -----------------------
# SILVER rare supplement (Train only)
# -----------------------
for d in tqdm(selected_silver, desc="Processing SILVER rare supplement"):
    ok = process_record_to_yolo(d, SILVER_DIR / "images", "train")
    if ok:
        silver_train_count += 1

random.shuffle(train_files)

print("\n✅ Build dataset xong.")
print(f"Gold train images used: {gold_train_count}")
print(f"Gold val images used:   {gold_val_count}")
print(f"Silver train images used: {silver_train_count}")
print(f"Bad/Skipped records: {bad_records}")

print("\nTrain class distribution:")
for cls, cnt in sorted(class_counter_train.items(), key=lambda kv: kv[1]):
    print(f"  - {cls}: {cnt}")

print("\nVal class distribution:")
for cls, cnt in sorted(class_counter_val.items(), key=lambda kv: kv[1]):
    print(f"  - {cls}: {cnt}")

Processing GOLD VAL: 100%|██████████| 552/552 [00:01<00:00, 425.20it/s]
Processing SILVER rare supplement: 0it [00:00, ?it/s]


✅ Build dataset xong.
Gold train images used: 2208
Gold val images used:   414
Silver train images used: 0
Bad/Skipped records: 2

Train class distribution:
  - graph: 165
  - image: 654
  - table: 731
  - annotation: 1085
  - formula: 3628
  - printed: 5467
  - handwritten: 24389

Val class distribution:
  - graph: 13
  - table: 137
  - image: 150
  - annotation: 227
  - formula: 797
  - printed: 809
  - handwritten: 5688


In [7]:
# =========================================================
# CELL 6: GHI train.txt / val.txt / yaml
# =========================================================
train_txt = OUT_ROOT / "train.txt"
val_txt   = OUT_ROOT / "val.txt"

with open(train_txt, "w", encoding="utf-8") as f:
    for fname in train_files:
        f.write(str(OUT_ROOT / "images" / fname) + "\n")

with open(val_txt, "w", encoding="utf-8") as f:
    for fname in val_files:
        f.write(str(OUT_ROOT / "images" / fname) + "\n")

yaml_path = "/kaggle/working/rukopys_dataset_v3.yaml"
data_config = {
    "path": str(OUT_ROOT),
    "train": "train.txt",
    "val": "val.txt",
    "nc": 7,
    "names": ["handwritten", "printed", "formula", "table", "annotation", "image", "graph"]
}

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_config, f, sort_keys=False)

# Giữ nguyên bản vá hoang dã .yaml.yaml
shutil.copy(yaml_path, yaml_path + ".yaml")

print(f"Train images: {len(train_files)}")
print(f"Val images:   {len(val_files)}")
print("YAML written:", yaml_path)
print("YAML duplicate:", yaml_path + ".yaml")

Train images: 2208
Val images:   414
YAML written: /kaggle/working/rukopys_dataset_v3.yaml
YAML duplicate: /kaggle/working/rukopys_dataset_v3.yaml.yaml


In [8]:
# =========================================================
# CELL 7: SETUP DOCLAYOUT-YOLO + PATCH
# =========================================================
import sys
import os
from types import ModuleType

%cd /kaggle/working

if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git

%cd /kaggle/working/DocLayout-YOLO
!pip install -q -e .

# --- Patch check_amp ---
checks_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/checks.py"
if os.path.exists(checks_file):
    with open(checks_file, "r", encoding="utf-8") as f:
        code = f.read()
    if "def check_amp(model):" in code and "return True" not in code.split("def check_amp(model):")[1][:40]:
        code = code.replace("def check_amp(model):", "def check_amp(model):\n    return True\n")
        with open(checks_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Patched check_amp()")

# --- Patch strip_optimizer / torch.load for PyTorch 2.6 ---
torch_utils_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/torch_utils.py"
if os.path.exists(torch_utils_file):
    with open(torch_utils_file, "r", encoding="utf-8") as f:
        code = f.read()

    old_load = 'x = torch.load(f, map_location=torch.device("cpu"))'
    new_load = 'x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)'

    if old_load in code:
        code = code.replace(old_load, new_load)
        with open(torch_utils_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Patched strip_optimizer torch.load(..., weights_only=False)")

# --- Patch G2L_CRM Fuse Bug ---
g2l_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/nn/modules/g2l_crm.py"
if os.path.exists(g2l_file):
    with open(g2l_file, "r", encoding="utf-8") as f:
        code = f.read()
    
    changed = False
    if "bn = self.dcv.bn" in code:
        code = code.replace("bn = self.dcv.bn", "bn = getattr(self.dcv, 'bn', None)")
        changed = True
        
    old_return_call = "return act(bn(F.conv2d(x, weight, stride=1, padding=padding, dilation=dilation)))"
    if old_return_call in code:
        new_return_call = "bias = getattr(self.dcv.conv, 'bias', None)\n        out = F.conv2d(x, weight, bias=bias, stride=1, padding=padding, dilation=dilation)\n        return act(bn(out) if bn is not None else out)"
        code = code.replace(old_return_call, new_return_call)
        changed = True

    if changed:
        with open(g2l_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Patched G2L_CRM for fuse compatibility (handled None bn & fused bias)")

!pip uninstall ray -y -q

if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

print("✅ DocLayout-YOLO setup xong.")
# --- Patch NumPy 2.0 compatibility trong metrics.py ---
metrics_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/metrics.py"
if os.path.exists(metrics_file):
    with open(metrics_file, "r", encoding="utf-8") as f:
        code = f.read()
    
    if "np.trapz" in code:
        code = code.replace("np.trapz", "np.trapezoid")
        with open(metrics_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Patched np.trapz thành np.trapezoid để tương thích NumPy 2.0")

/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 237 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 25.75 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.6 MB/s eta 0:00:00
  Building editable for doclayout_yolo (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 w

In [9]:
# =========================================================
# CELL 8: PRETRAIN + TRAIN PATHS
# =========================================================
from pathlib import Path

DATA_YAML = "/kaggle/working/rukopys_dataset_v3.yaml"
EXPERIMENT_ROOT = Path("/kaggle/working/DocLayout-YOLO/runs/train")
MODEL_NAME = "doclayout_yolo_small"

CKPT_PATH = hf_hub_download(
    repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
    filename="doclayout_yolo_docstructbench_imgsz1024.pt"
)

print("MODEL_NAME:", MODEL_NAME)
print("CKPT_PATH:", CKPT_PATH)
print("DATA_YAML:", DATA_YAML)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)

doclayout_yolo_docstructbench_imgsz1024.(…):   0%|          | 0.00/40.7M [00:00<?, ?B/s]

MODEL_NAME: doclayout_yolo_small
CKPT_PATH: /root/.cache/huggingface/hub/models--juliozhao--DocLayout-YOLO-DocStructBench/snapshots/8c3299a30b8ff29a1503c4431b035b93220f7b11/doclayout_yolo_docstructbench_imgsz1024.pt
DATA_YAML: /kaggle/working/rukopys_dataset_v3.yaml
EXPERIMENT_ROOT: /kaggle/working/DocLayout-YOLO/runs/train


In [10]:
# =========================================================
# CELL 9: TRAIN (DUNG THAM SO CLI THUC SU DUOC HO TRO)
# =========================================================
%cd /kaggle/working/DocLayout-YOLO

print("\n🔥 KHOI DONG HUAN LUYEN V3 + MOSAIC NHE 🔥")

!WANDB_MODE=disabled RAY_DISABLE_TUNE=1 python train.py \
  --data {DATA_YAML} \
  --model {MODEL_NAME} \
  --epoch 100 \
  --image-size 1024 \
  --batch-size 8 \
  --mosaic 0.30 \
  --project rukopys_v3_clean_gold_plus_rare_silver \
  --optimizer Adam \
  --lr0 0.001 \
  --warmup-epochs 1.0 \
  --patience 8 \
  --pretrain {CKPT_PATH} \
  --device 0,1 \
  --workers 8

/kaggle/working/DocLayout-YOLO

🔥 KHOI DONG HUAN LUYEN V3 + MOSAIC NHE 🔥
New https://pypi.org/project/doclayout_yolo/0.0.4 available 😃 Update with 'pip install -U doclayout_yolo'
Ultralytics YOLOv0.0.2 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                            CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/root/.cache/huggingface/hub/models--juliozhao--DocLayout-YOLO-DocStructBench/snapshots/8c3299a30b8ff29a1503c4431b035b93220f7b11/doclayout_yolo_docstructbench_imgsz1024.pt, data=/kaggle/working/rukopys_dataset_v3.yaml.yaml, epochs=100, time=None, patience=8, batch=8, imgsz=1024, save=True, save_period=10, val_period=1, cache=False, device=0,1, workers=8, project=rukopys_v3_clean_gold_plus_rare_silver, name=rukopys_dataset_v3.yaml_epoch100_imgsz1024_bs8_pretrain_unknown, exist_ok=False, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos

In [11]:
# =========================================================
# CELL 10: SAVE WEIGHTS (SAU KHI XONG 1 FOLD)
# =========================================================
import shutil
from pathlib import Path

out_dir = Path("/kaggle/working/DocLayout-YOLO/runs/train") / f"run_fold_{VAL_FOLD}" / "weights"
if not out_dir.exists():
    out_dir = EXPERIMENT_ROOT / f"run_fold_{VAL_FOLD}" / "weights"

if out_dir.exists():
    last_pt = out_dir / "last.pt"
    best_pt = out_dir / "best.pt"
    
    if last_pt.exists():
        shutil.copy(last_pt, f"/kaggle/working/doclayout_yolo_fold_{VAL_FOLD}_last.pt")
        print(f"✅ Saved doclayout_yolo_fold_{VAL_FOLD}_last.pt")
    if best_pt.exists():
        shutil.copy(best_pt, f"/kaggle/working/doclayout_yolo_fold_{VAL_FOLD}_best.pt")
        print(f"✅ Saved doclayout_yolo_fold_{VAL_FOLD}_best.pt")
else:
    print("❌ Không tìm thấy thư mục weights. Cần kiểm tra lại quá trình train.")

# Cuối cùng, có thể tuỳ chọn zip thư mục dataset nếu cần download
# !zip -r /kaggle/working/yolo_dataset.zip /kaggle/working/doclayout_yolo_dataset


❌ Không tìm thấy thư mục weights. Cần kiểm tra lại quá trình train.
